In [ ]:
import pandas as pd

df = pd.read_csv('uci-secom.csv')
df.shape

(1567, 592)

In [ ]:
# 1) 결측치 비율 확인 - 어떤 센서가 얼마나 비어있는지
missing_ratio = df.isnull().mean().sort_values(ascending=False)
print(missing_ratio.head(20))

293    0.911934
292    0.911934
157    0.911934
158    0.911934
492    0.855775
220    0.855775
85     0.855775
358    0.855775
518    0.649649
382    0.649649
245    0.649649
244    0.649649
383    0.649649
384    0.649649
246    0.649649
517    0.649649
110    0.649649
109    0.649649
516    0.649649
111    0.649649
dtype: float64


In [ ]:
# 2) 불량/정상 비율 확인
print(df['Pass/Fail'].value_counts())
print(df['Pass/Fail'].value_counts(normalize=True))


Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64
Pass/Fail
-1    0.933631
 1    0.066369
Name: proportion, dtype: float64


In [ ]:
# 1) 결측 비율 40% 이상인 열 제거
cols_to_drop = missing_ratio[missing_ratio > 0.4].index
df = df.drop(columns=cols_to_drop)

# 2) 남은 결측치는 중앙값으로 채우기 (베이스라인)
df = df.fillna(df.median(numeric_only=True))

df.shape

(1567, 560)

In [ ]:
X = df.drop(columns=['Time', 'Pass/Fail'])
y = df['Pass/Fail'].replace(-1, 0)  # XGBoost 위해 0/1로 변환

X.shape, y.value_counts()

((1567, 558),
 Pass/Fail
 0    1463
 1     104
 Name: count, dtype: int64)

In [ ]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.05)
X_selected = selector.fit_transform(X)
selected_features = X.columns[selector.get_support()]
X = pd.DataFrame(X_selected, columns=selected_features)

X.shape


(1567, 267)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((1253, 267), (314, 267))

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

y_train_res.value_counts()

,count
Pass/Fail,
0,1170
1,1170


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape

((2340, 267), (314, 267))

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'scale_pos_weight': [1, 5, 10]
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    scoring='recall',
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train_res)

grid_search.best_params_

Fitting 3 folds for each of 81 candidates, totalling 243 fits


{'learning_rate': 0.01,
 'max_depth': 3,
 'n_estimators': 50,
 'scale_pos_weight': 5}

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.02      0.05       293
           1       0.07      1.00      0.13        21

    accuracy                           0.09       314
   macro avg       0.53      0.51      0.09       314
weighted avg       0.94      0.09      0.05       314



In [ ]:
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'scale_pos_weight': [1]   # SMOTE로 이미 균형 맞췄으므로 추가 가중치 제거
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    scoring='f1',   # recall 단독 대신 f1로 변경 (precision-recall 균형)
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train_res)
grid_search.best_params_

Fitting 3 folds for each of 27 candidates, totalling 81 fits


{'learning_rate': 0.1,
 'max_depth': 7,
 'n_estimators': 100,
 'scale_pos_weight': 1}

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.99      0.96       293
           1       0.00      0.00      0.00        21

    accuracy                           0.92       314
   macro avg       0.47      0.49      0.48       314
weighted avg       0.87      0.92      0.90       314



In [ ]:
param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [50, 100],
    'scale_pos_weight': [2, 3, 5]
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train_res)
grid_search.best_params_

Fitting 3 folds for each of 24 candidates, totalling 72 fits


{'learning_rate': 0.1,
 'max_depth': 5,
 'n_estimators': 100,
 'scale_pos_weight': 2}

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.97      0.95       293
           1       0.10      0.05      0.06        21

    accuracy                           0.91       314
   macro avg       0.52      0.51      0.51       314
weighted avg       0.88      0.91      0.89       314



In [ ]:
import numpy as np

# 지금까지 학습된 모델에서 변수 중요도 추출
importances = best_model.feature_importances_
feature_names = X.columns

top_features = pd.Series(importances, index=feature_names).sort_values(ascending=False)
top_features.head(30)

,0
486,0.039377
31,0.023733
430,0.023035
511,0.020694
388,0.019345
59,0.018814
207,0.018387
33,0.018070
490,0.016040
487,0.012237


In [ ]:
top30 = top_features.head(30).index
X_top = X[top30]
# 다시 train/test 분할부터
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_top, y, test_size=0.2, random_state=42, stratify=y
)
smote2 = SMOTE(random_state=42)
X_train2_res, y_train2_res = smote2.fit_resample(X_train2, y_train2)
scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2_res)
X_test2_scaled = scaler2.transform(X_test2)
X_train2_scaled.shape, X_test2_scaled.shape

((2340, 30), (314, 30))

In [ ]:
param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [50, 100],
    'scale_pos_weight': [2, 3, 5]
}

grid_search2 = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search2.fit(X_train2_scaled, y_train2_res)
grid_search2.best_params_


Fitting 3 folds for each of 24 candidates, totalling 72 fits


{'learning_rate': 0.1,
 'max_depth': 5,
 'n_estimators': 100,
 'scale_pos_weight': 2}

In [ ]:
best_model2 = grid_search2.best_estimator_
y_pred2 = best_model2.predict(X_test2_scaled)

print(classification_report(y_test2, y_pred2))

              precision    recall  f1-score   support

           0       0.95      0.93      0.94       293
           1       0.23      0.29      0.26        21

    accuracy                           0.89       314
   macro avg       0.59      0.61      0.60       314
weighted avg       0.90      0.89      0.89       314



In [ ]:
top50 = top_features.head(50).index
X_top50 = X[top50]

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X_top50, y, test_size=0.2, random_state=42, stratify=y
)

smote3 = SMOTE(random_state=42)
X_train3_res, y_train3_res = smote3.fit_resample(X_train3, y_train3)

scaler3 = StandardScaler()
X_train3_scaled = scaler3.fit_transform(X_train3_res)
X_test3_scaled = scaler3.transform(X_test3)

X_train3_scaled.shape, X_test3_scaled.shape

((2340, 50), (314, 50))

In [ ]:
param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [50, 100],
    'scale_pos_weight': [2, 3, 5]
}

grid_search3 = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search3.fit(X_train3_scaled, y_train3_res)
grid_search3.best_params_

Fitting 3 folds for each of 24 candidates, totalling 72 fits


{'learning_rate': 0.1,
 'max_depth': 5,
 'n_estimators': 100,
 'scale_pos_weight': 2}

In [ ]:
best_model3 = grid_search3.best_estimator_
y_pred3 = best_model3.predict(X_test3_scaled)

print(classification_report(y_test3, y_pred3))

              precision    recall  f1-score   support

           0       0.94      0.94      0.94       293
           1       0.11      0.10      0.10        21

    accuracy                           0.89       314
   macro avg       0.52      0.52      0.52       314
weighted avg       0.88      0.89      0.88       314



In [ ]:
from imblearn.combine import SMOTEENN

smote_enn = SMOTEENN(random_state=42)
X_train4_res, y_train4_res = smote_enn.fit_resample(X_train2, y_train2)  # 30개 변수 기준

y_train4_res.value_counts()

,count
Pass/Fail,
1,1140
0,773


In [ ]:
param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.1],
    'n_estimators': [100],
    'scale_pos_weight': [1, 2]   # 이미 ENN이 정리했으니 추가 가중치는 약하게
}

grid_search4 = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search4.fit(X_train4_res, y_train4_res)
grid_search4.best_params_

Fitting 3 folds for each of 4 candidates, totalling 12 fits


{'learning_rate': 0.1,
 'max_depth': 5,
 'n_estimators': 100,
 'scale_pos_weight': 1}

In [ ]:
best_model4 = grid_search4.best_estimator_
y_pred4 = best_model4.predict(X_test2_scaled)  # 30개 변수 기준 Test셋 그대로 사용

print(classification_report(y_test2, y_pred4))

              precision    recall  f1-score   support

           0       0.93      1.00      0.97       293
           1       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
top15 = top_features.head(15).index
X_top15 = X[top15]

X_train5, X_test5, y_train5, y_test5 = train_test_split(
    X_top15, y, test_size=0.2, random_state=42, stratify=y
)

smote5 = SMOTE(random_state=42)
X_train5_res, y_train5_res = smote5.fit_resample(X_train5, y_train5)

scaler5 = StandardScaler()
X_train5_scaled = scaler5.fit_transform(X_train5_res)
X_test5_scaled = scaler5.transform(X_test5)

X_train5_scaled.shape, X_test5_scaled.shape

((2340, 15), (314, 15))

In [ ]:
param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [50, 100],
    'scale_pos_weight': [2, 3, 5]
}

grid_search5 = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search5.fit(X_train5_scaled, y_train5_res)
grid_search5.best_params_

Fitting 3 folds for each of 24 candidates, totalling 72 fits


{'learning_rate': 0.1,
 'max_depth': 5,
 'n_estimators': 100,
 'scale_pos_weight': 2}

In [ ]:
best_model5 = grid_search5.best_estimator_
y_pred5 = best_model5.predict(X_test5_scaled)

print(classification_report(y_test5, y_pred5))

              precision    recall  f1-score   support

           0       0.95      0.88      0.91       293
           1       0.15      0.29      0.20        21

    accuracy                           0.84       314
   macro avg       0.55      0.58      0.56       314
weighted avg       0.89      0.84      0.87       314

